# AgentCore Long Memory Example

This notebook demonstrates how to use Bedrock AgentCore Memory with LangGraph to create a nutrition assistant that remembers user preferences across conversations.


In [ ]:
import os
import logging
import uuid

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import before_model, after_model
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
from langgraph_checkpoint_aws import AgentCoreMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from custom_memory_prompts import consolidation_prompt, extraction_prompt


## Configuration

Set up the region, logging, memory name, and API keys.


In [ ]:
region = os.getenv('AWS_REGION', 'us-east-1')
logging.getLogger("math-agent").setLevel(logging.DEBUG)

memory_name = "NutritionAssistant"
BEDROCK_MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"



# Get execution role ARN from environment variable or use default
memory_execution_role_arn = os.getenv(
    'MEMORY_EXECUTION_ROLE_ARN',
    'arn:aws:iam::YOUR_ACCOUNT:role/BedrockAgentCoreExecutionRole'  # Default role name
)


## Initialize Memory Client

Create the memory client and set up the memory with custom strategies for capturing nutrition preferences.


In [ ]:
client = MemoryClient(region_name=region)

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant",
    memory_execution_role_arn=memory_execution_role_arn,
    strategies=[
        {
            StrategyType.CUSTOM.value: {
                "name": "NutritionPreferences",
                "description": "Captures customer food preferences and behavior",
                "namespaces": ["/{actorId}/preferences"],
                "configuration": {
                    "userPreferenceOverride": {
                        "extraction": {
                            "appendToPrompt": extraction_prompt,
                            "modelId": BEDROCK_MODEL_ID,  # Must be a Bedrock model ID
                        },
                        "consolidation": {
                            "appendToPrompt": consolidation_prompt,
                            "modelId": BEDROCK_MODEL_ID,  # Must be a Bedrock model ID
                        }
                    }
                }
            }
        },
    ]
)
memory_id = memory["id"]
print(f"Memory created/retrieved with ID: {memory_id}")


## Initialize Store and LLM

Set up the AgentCore Memory Store for long-term memory and initialize the LLM.


In [ ]:
# Initialize the store to enable long term memory saving and retrieval
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Initialize OpenAI LLM for the main conversation
# Note: The memory operations use Bedrock (configured above), but the main LLM can use OpenAI
llm = init_chat_model(BEDROCK_MODEL_ID, model_provider="bedrock_converse", region_name=region)

# Note: We'll use client.create_event() directly in hooks to trigger extraction


## Define Hooks

Create pre-model and post-model hooks to save and retrieve messages from AgentCore Memory.


In [ ]:
@before_model
def pre_model_hook(state, runtime: Runtime):
    """Hook that runs pre-LLM invocation to save the latest human message"""
    print("pre_model_hook")
    # Access config from runtime.context
    # The context should contain the configurable values (actor_id, thread_id) passed during invocation
    context = runtime.context
    
    if context is None:
        # If context is None, use defaults
        print("Warning: runtime.context is None. Using defaults.")
        actor_id = "default-actor"
        thread_id = "default-thread"
        print(f"Using default actor_id={actor_id} and thread_id={thread_id}")
    elif isinstance(context, dict):
        # Context is a dict with configurable values directly
        actor_id = context.get("actor_id")
        thread_id = context.get("thread_id")
    else:
        # Context is an object, try attribute access
        actor_id = getattr(context, "actor_id", None)
        thread_id = getattr(context, "thread_id", None)
    
    if not actor_id or not thread_id:
        raise ValueError(f"Missing actor_id or thread_id. Context: {context}")
    
    store = runtime.store
    
    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)
    
    messages = state.get("messages", [])
    print(f"Pre model hook messages: {messages}")
    # Save the last human message we see before LLM invocation
    last_human_msg = None
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            last_human_msg = msg
            break
    
    # Retrieve user preferences based on the last message and append to state
    if last_human_msg:
        user_preferences_namespace = (actor_id, "preferences")
        preferences = store.search(user_preferences_namespace, query=last_human_msg.content, limit=5)
        
        # Construct another AI message to add context before the current message
        if preferences:
            context_items = [pref.value for pref in preferences]
            context_message = AIMessage(
                content=f"[User Context: {', '.join(str(item) for item in context_items)}]"
            )
            # Insert the context message before the last human message
            return {"messages": messages[:-1] + [context_message, messages[-1]]}
    
    return {"messages": messages}




In [ ]:
@after_model
def post_model_hook(state, runtime: Runtime):
    """Hook that runs post-LLM invocation to save the latest human message"""
    print("post_model_hook")
    # Access config from runtime.context
    # The context should contain the configurable values (actor_id, thread_id) passed during invocation
    context = runtime.context
    
    if context is None:
        # If context is None, use defaults
        print("Warning: runtime.context is None. Using defaults.")
        actor_id = "default-actor"
        thread_id = "default-thread"
        print(f"Using default actor_id={actor_id} and thread_id={thread_id}")
    elif isinstance(context, dict):
        # Context is a dict with configurable values directly
        actor_id = context.get("actor_id")
        thread_id = context.get("thread_id")
    else:
        # Context is an object, try attribute access
        actor_id = getattr(context, "actor_id", None)
        thread_id = getattr(context, "thread_id", None)
    
    if not actor_id or not thread_id:
        raise ValueError(f"Missing actor_id or thread_id. Context: {context}")
    
    store = runtime.store

    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)
    
    messages = state.get("messages", [])
    print(f"Post model hook messages: {messages}")
    # Save the LLMs response to AgentCore Memory
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break
    
    return {"messages": messages}


## Create the Agent Graph

Create the LangGraph agent with the memory store and hooks.


In [ ]:
graph = create_agent(
    llm,
    tools=[], # No additional tools needed for this example
    checkpointer=InMemorySaver(), # For conversation state management
    store=store, # Store for long-term memory
    middleware=[pre_model_hook, post_model_hook]  # Middleware hooks for memory operations
)


## Set Up Configuration

Configure the actor ID and thread ID for the conversation.


In [ ]:
actor_id = "user-2"
config = {
    "configurable": {
        "thread_id": "session-1", # REQUIRED: This maps to Bedrock AgentCore session_id under the hood
        "actor_id": actor_id, # REQUIRED: This maps to Bedrock AgentCore actor_id under the hood
    }
}


## Helper Function

Create a helper function to run the agent and pretty print the output.


In [ ]:
# Helper function to pretty print agent output while running
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    # Use stream instead of invoke to get events properly
    # Pass configurable values through context so middleware can access them
    context = config.get("configurable", {}) if isinstance(config, dict) else {}
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        context=context,  # Pass configurable values as context so runtime.context has access to them
        stream_mode="values",
    )
    for event in events:
        # event might be a dict with messages or the state itself
        if isinstance(event, dict):
            if "messages" in event:
                for msg in event["messages"]:
                    # Check if we've already printed this message
                    if id(msg) not in printed_ids:
                        msg.pretty_print()
                        printed_ids.add(id(msg))
            else:
                # If event is the state dict directly, check for messages
                messages = event.get("messages", [])
                for msg in messages:
                    if id(msg) not in printed_ids:
                        msg.pretty_print()
                        printed_ids.add(id(msg))


## Example 1: First Conversation

Run the first conversation about cooking salmon.


In [ ]:
prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)


## Search User Preferences

Search the stored preferences to see what was captured.


In [ ]:
# Search our user preferences namespace
search_namespace = (actor_id, "preferences")
result = store.search(search_namespace, query="food", limit=3)
print(f"Preferences namespace result: {result}")


## Example 2: Second Conversation (New Session)

Run a second conversation in a new session. The agent should remember preferences from the previous conversation.


In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2", # New session ID
        "actor_id": actor_id, # Same actor ID
    }
}

run_agent("Today's a new day, what should I make for dinner tonight?", config)
